# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTD504/flyrank-ai-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [1]:
# Setup DuckDB and Hugging Face authentication
!pip install -q duckdb huggingface_hub pandas

import duckdb
from google.colab import userdata

# Retrieve HF Token securely from Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    hf_token = getpass.getpass("Enter your HF Read Token: ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}');")

# Target a mid-panel month partition (e.g., March 2026) for contract iteration[cite: 1]
MID_PANEL_MONTH = "2026-03"
TABLE_PATH = f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MID_PANEL_MONTH}/*.parquet')"

print(f"✅ Successfully connected DuckDB to Hugging Face Warehouse ({MID_PANEL_MONTH})!")

✅ Successfully connected DuckDB to Hugging Face Warehouse (2026-03)!


## 1. Unit of analysis + time window

### 1. Unit of Analysis (Grain)
* **Definition:** One row represents a daily performance record for a single pseudonymized article (`content_hash_id`) belonging to a specific client (`client_hash_id`) on a given date (`report_date`).
* **Composite Primary Key:** `(client_hash_id, content_hash_id, report_date)`.
* **Grain Rule:** Every unique combination of `client_hash_id`, `content_hash_id`, and `report_date` must appear at most once per daily performance snapshot.

### 2. Time Window
* **Mid-panel Evaluation Window:** A single-month partition snapshot `2026-03` (`report_date` ranging from `2026-03-01` to `2026-03-31`) used for iterative query development and contract validation.
* **Sealed Holdout Window:** The final panel month (`2026-06` / `_sample` table) is strictly reserved as an un-touched test window to prevent label logic contamination during feature development.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 1: Verify the Unit of Analysis (Grain), Row Count, and Date Range
query_s1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT content_hash_id) AS total_articles,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {TABLE_PATH}
"""

df_s1 = con.sql(query_s1).df()
print("=== Section 1 Verification: Grain & Time Window ===")
print(df_s1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Section 1 Verification: Grain & Time Window ===
   total_rows  total_clients  total_articles min_report_date max_report_date
0     9841378             55          331437      2026-03-01      2026-03-31


## 2. Fields: feature / label / context / excluded

All touched fields are sorted into four mutually exclusive contract buckets based on their operational availability at decision time:

1. **Context Bucket (Identifiers & Grouping):**
   * `client_hash_id`, `content_hash_id`, `report_date`: Pseudonymized keys used exclusively for table joins, time-series aggregation, and entity-aware splitting (`GroupKFold` by `client_hash_id`). These identifiers are never fed into the model as predictive features.

2. **Feature Bucket (Knowable BEFORE the decision moment):**
   * **Search Performance:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (historical organic search performance metrics recorded up to the prediction cutoff).
   * **User Engagement:** `ga4_sessions`, `ga4_engagement_rate`, `ga4_scroll_rate` (historical user behavior metrics).
   * **Content Freshness:** `content_age_days`, `days_since_last_update`, `freshness_tier` (staleness indicators knowable at decision time).
   * **Instrumentation Quality Flag:** `ga4_data_available` (boolean flag indicating whether active GA4 tracking was enabled vs missing/NULL, ensuring non-instrumented pages are not mistaken for zero-engagement content).

3. **Label / Proxy Bucket (Target to predict):**
   * `target_is_decaying`: Binary target (`1` if an article exhibits severe organic performance decay—comparing current evaluation window clicks against previous periods—and `0` otherwise).

4. **Excluded Bucket (With Explicit Reasons):**
   * **Derived Trend Metrics (`trend_direction` & `trend_pct`):** **EXCLUDED** due to direct **Label Leakage**. In summary snapshots, target labels are computed directly from these trend indicators. Feeding them as features exposes future outcome information to the model.
   * **Product Decision Flags (`health_score`, `needs_ctr_fix`, `is_quick_win`):** **EXCLUDED** to prevent circular learning. These fields represent output rules from an existing decision system rather than raw environmental signals.

In [5]:
# Query 2: Inspect a sample row to verify column availability and bucket categorization
query_s2 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_data_available
FROM {TABLE_PATH}
LIMIT 5
"""

df_s2 = con.sql(query_s2).df()
print("=== Section 2 Verification: Sample Column Inspection ===")
print(df_s2)

=== Section 2 Verification: Sample Column Inspection ===
            client_hash_id           content_hash_id report_date  \
0  client_73cda7b4e4f265ea  content_b7e512995f79d5a6  2026-03-01   
1  client_73cda7b4e4f265ea  content_05597932fe4da067  2026-03-01   
2  client_73cda7b4e4f265ea  content_7a105f548d9c6916  2026-03-01   
3  client_73cda7b4e4f265ea  content_905aa32a0230694e  2026-03-01   
4  client_73cda7b4e4f265ea  content_a3ea9792f793ec72  2026-03-01   

   gsc_impressions  gsc_clicks  gsc_avg_position  ga4_sessions  \
0               20           0          3.350000          <NA>   
1                1           0          0.000000          <NA>   
2              125           1          4.928000          <NA>   
3                7           0          4.000000          <NA>   
4               11           0          2.272727          <NA>   

   ga4_data_available  
0                <NA>  
1                <NA>  
2                <NA>  
3                <NA>  
4                

## 3. Verify it with queries (grain, counts, missing values, windows)

### Contract Verification Suite

To validate the contract claims declared in Section 1 and Section 2, three targeted DuckDB verification queries are executed on the mid-panel evaluation month (`2026-03`):

1. **Grain Probe (Composite Primary Key Uniqueness):** Runs a `GROUP BY client_hash_id, content_hash_id, report_date HAVING COUNT(*) > 1` query to prove zero duplicate rows exist.
2. **Volume & Date Window Span:** Verifies total row count, distinct clients, distinct articles, and exact boundary dates (`2026-03-01` to `2026-03-31`).
3. **Availability & Quality Audit (`IS_TRUE` Filter):** Evaluates instrumentation coverage using `ga4_data_available IS TRUE` vs `FALSE` / `NULL` to quantify clean Analytics signal availability.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 1: Grain Probe - Verify composite key uniqueness (Expect 0 duplicates)
query_grain = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS dup_count
FROM {TABLE_PATH}
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
"""

df_grain = con.sql(query_grain).df()
print("=== Probe 1: Grain Duplicate Probe (Should return 0 rows) ===")
print(f"Duplicate rows detected: {len(df_grain)}")
if len(df_grain) > 0:
    print(df_grain)

# Query 2: Row count & exact date span boundaries
query_volume = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_articles
FROM {TABLE_PATH}
"""

df_volume = con.sql(query_volume).df()
print("\n=== Probe 2: Volume & Date Span Verification ===")
print(df_volume)

# Query 3: Availability Check using IS_TRUE filter
query_availability = f"""
SELECT
    COALESCE(ga4_data_available, FALSE) IS TRUE AS is_ga4_available,
    COUNT(*) AS row_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM {TABLE_PATH}
GROUP BY COALESCE(ga4_data_available, FALSE) IS TRUE
"""

df_avail = con.sql(query_availability).df()
print("\n=== Probe 3: GA4 Availability Breakdown (IS_TRUE filter) ===")
print(df_avail)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Probe 1: Grain Duplicate Probe (Should return 0 rows) ===
Duplicate rows detected: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Probe 2: Volume & Date Span Verification ===
   total_rows min_report_date max_report_date  distinct_clients  \
0     9841378      2026-03-01      2026-03-31                55   

   distinct_articles  
0             331437  


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Probe 3: GA4 Availability Breakdown (IS_TRUE filter) ===
   is_ga4_available  row_count  percentage
0             False    9427412       95.79
1              True     413966        4.21


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Named Data Slice Limitations

1. **Instrumentation & GA4 Availability Bias:** Only 4.21% of total daily performance rows in the `2026-03` partition carry active GA4 data (`ga4_data_available IS TRUE`). The remaining 95.79% of rows are missing or un-instrumented. Applying features derived from GA4 metrics without checking `ga4_data_available` risks treating untracked pages as zero-engagement content.
2. **Unbalanced Client History Depth:** Client tracking start dates (`gsc_data_start`) vary across the dataset. Global calendar rolling windows can introduce cold-start bias for clients onboarded midway through the observation frame.
3. **Observational Correlation vs. Causality:** The warehouse records observational search performance metrics. Observed traffic declines reflect correlated intent shifts or competitive dynamics rather than direct causal proof of content decay.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 4: Audit GA4 instrumentation breakdown and verify missingness vs availability flag
query_s4 = f"""
SELECT
    COALESCE(ga4_data_available, FALSE) IS TRUE AS ga4_is_active,
    COUNT(*) AS total_records,
    COUNT(ga4_sessions) AS non_null_sessions,
    ROUND(100.0 * COUNT(ga4_sessions) / COUNT(*), 2) AS pct_non_null_sessions
FROM {TABLE_PATH}
GROUP BY COALESCE(ga4_data_available, FALSE) IS TRUE
"""

df_s4 = con.sql(query_s4).df()
print("=== Section 4 Verification: GA4 Data Limits & Availability Audit ===")
print(df_s4)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Section 4 Verification: GA4 Data Limits & Availability Audit ===
   ga4_is_active  total_records  non_null_sessions  pct_non_null_sessions
0          False        9427412            6408671                  67.98
1           True         413966             413966                 100.00


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.